In [1]:
import sys
import pandas as pd
import pyarrow as pa

print("Python:", sys.executable)
print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)

Python: c:\Users\MY NGOC\Documents\MAU_TACAS_68803\DA631E_ArtificialIntelligenceForDataScience\.venv\Scripts\python.exe
Pandas: 3.0.6
PyArrow: 25.0.1


In [2]:
#!pip install pandas numpy matplotlib missingno
#%pip install scikit-learn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import missingno as msno

import os
from pathlib import Path

#import sklearn.preprocessing

pd.options.display.max_rows=1000

import sys
import json
from IPython.display import display


def find_project_root(start: Path = Path.cwd()) -> Path:
    """Find the parent directory containing src/data.py."""
    for candidate in (start, *start.parents):
        if (candidate / "src" / "data.py").is_file():
            return candidate

    raise FileNotFoundError("Could not find src/data.py")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
    


#from data import load_data, clean_prices, clean_stock_list, clean_financials
from config import ProjectConfig, ProjectPaths
from features import (
    add_price_features
    ,assert_causal_feature_stability
    ,feature_missingness
    ,infer_feature_columns
    ,join_financials_asof
    ,join_stock_metadata
)


from modeling import save_json

config = ProjectConfig()
paths = ProjectPaths(PROJECT_ROOT)
paths.ensure_output_dirs

print("Project root:", PROJECT_ROOT)
print("Imported load_data successfully")

print("Imported libraries sucessfully! yay")

Project root: c:\Users\MY NGOC\Documents\MAU_TACAS_68803\DA631E_ArtificialIntelligenceForDataScience\Project_1_SupervisedLearning
Imported load_data successfully
Imported libraries sucessfully! yay


In [3]:

prices = pd.read_parquet(paths.interim / "stock_prices_clean.parquet")
stock_list = pd.read_parquet(paths.interim / "stock_list_clean.parquet")
financials = pd.read_parquet(paths.interim / "financials_clean.parquet")

print(prices.shape, stock_list.shape, financials.shape)

(2332531, 21) (4417, 12) (87293, 23)


## 1. Build causal price and volume features

In [4]:
print(type(prices))
print(prices.shape)
print(prices.columns.tolist())
display(prices.head())

<class 'pandas.DataFrame'>
(2332531, 21)
['RowId', 'Date', 'SecuritiesCode', 'Open', 'High', 'Low', 'Close', 'Volume', 'AdjustmentFactor', 'ExpectedDividend', 'SupervisionFlag', 'Target', 'PartialOHLCFlag', 'NoTradeFlag', 'MarketWideNoTradeFlag', 'StockSpecificNoTradeFlag', 'AdjustmentEventFlag', 'HasExpectedDividend', 'LastTradeDate', 'DaysSinceLastTrade', 'CloseForFeatures']


,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,...,Target,PartialOHLCFlag,NoTradeFlag,MarketWideNoTradeFlag,StockSpecificNoTradeFlag,AdjustmentEventFlag,HasExpectedDividend,LastTradeDate,DaysSinceLastTrade,CloseForFeatures
0,20170104_1301,2017-01-04,1301,2734.0,2755.0,2730.0,2742.0,31400,1.0,0.0,...,0.000730,0,0,0,0,0,0,2017-01-04,0.0,2742.0
1,20170105_1301,2017-01-05,1301,2743.0,2747.0,2735.0,2738.0,17900,1.0,0.0,...,0.002920,0,0,0,0,0,0,2017-01-05,0.0,2738.0
2,20170106_1301,2017-01-06,1301,2734.0,2744.0,2720.0,2740.0,19900,1.0,0.0,...,-0.001092,0,0,0,0,0,0,2017-01-06,0.0,2740.0
3,20170110_1301,2017-01-10,1301,2745.0,2754.0,2735.0,2748.0,24200,1.0,0.0,...,-0.005100,0,0,0,0,0,0,2017-01-10,0.0,2748.0
4,20170111_1301,2017-01-11,1301,2748.0,2752.0,2737.0,2745.0,9300,1.0,0.0,...,-0.003295,0,0,0,0,0,0,2017-01-11,0.0,2745.0


In [5]:
prices_features = add_price_features(prices)
display(prices_features.head())
print(prices_features.shape)

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,...,VolatilityRatio20To60,CloseToMovingAverage20Days,LogVolume,VolumeVs20Days,LogTurnover,Return1DayRankPct,Momentum20DaysRankPct,Volatility20DaysRankPct,Month,DayOfWeek
0,20170104_1301,2017-01-04,1301,2734.0,2755.0,2730.0,2742.0,31400,1.0,0.0,...,NaN,NaN,10.354595,NaN,18.271006,NaN,NaN,NaN,1,2
1,20170104_1332,2017-01-04,1332,568.0,576.0,563.0,571.0,2798500,1.0,0.0,...,NaN,NaN,14.844594,NaN,21.191983,NaN,NaN,NaN,1,2
2,20170104_1333,2017-01-04,1333,3150.0,3210.0,3140.0,3210.0,270800,1.0,0.0,...,NaN,NaN,12.509140,NaN,20.583162,NaN,NaN,NaN,1,2
3,20170104_1376,2017-01-04,1376,1510.0,1550.0,1510.0,1550.0,11300,1.0,0.0,...,NaN,NaN,9.332646,NaN,16.678568,NaN,NaN,NaN,1,2
4,20170104_1377,2017-01-04,1377,3270.0,3350.0,3270.0,3330.0,150800,1.0,0.0,...,NaN,NaN,11.923716,NaN,20.034437,NaN,NaN,NaN,1,2


(2332531, 42)


## Check stability of features

Regression test for time leakage. The function computes features once with history truncated at a cutoff and once with later rows appended, then compare the historical results exactly (wothin floating tolerance)

In [6]:
sample_codes = sorted(prices['SecuritiesCode'].unique())[:10]
stability_sample = prices.loc[prices['SecuritiesCode'].isin(sample_codes)].copy()

cutoff = stability_sample['Date'].drop_duplicates().sort_values().iloc[
    int(stability_sample['Date'].nunique()* 0.7)
]

assert_causal_feature_stability(stability_sample, cutoff)
print(f"Passed: appending rows after {cutoff.date()} did not alter pass features.")

Passed: appending rows after 2020-06-16 did not alter pass features.


## Join static metadata with a cardinality check

`validate="many_to_one"` turns the intended relationship into executable
documentation. If `stock_list` unexpectedly gains multiple rows per security,
the pipeline stops instead of multiplying observations.

In [7]:
with_stock_list = join_stock_metadata(prices_features=prices_features, stock_meta=stock_list)
assert len(with_stock_list) == len(prices_features)

metadata_coverage = pd.Series({
    column: with_stock_list[column].notna().mean()
    for column in ["33SectorCode", "NewMarketSegment", "MarketCapitalizationLog"]
    if column in with_stock_list
}, name="non_missing_rate")

display(metadata_coverage)

33SectorCode               1.0
NewMarketSegment           1.0
MarketCapitalizationLog    1.0
Name: non_missing_rate, dtype: float64

## Join financials backward in time

For each price row, `merge_asof(..., direction="backward")` chooses the most recent financial disclosure whose `FinancialAvailableDate <= Date`. The invariant is asserted after the join. 
`FinancialAgeDays` lets the model distinguish a recent report from stale information.


In [8]:
from numpy import median


features_table = join_financials_asof(prices_features=with_stock_list, financial_disclosure=financials)
assert len(features_table) == len(prices)
assert not features_table.duplicated(['Date', 'SecuritiesCode']).any()
assert not (features_table['FinancialAvailableDate'] > features_table['Date']).fillna(False).any()

financials_coverage = pd.Series({
    'rows_with_prior_financial_report': features_table['FinancialAvailableDate'].notna().mean()
    ,'median_financial_age_days': features_table['FinancialAgeDays'].median()
    ,'max_financial_age_days': features_table['FinancialAgeDays'].max()
}, name = 'value')

display(financials_coverage)

rows_with_prior_financial_report      0.980084
median_financial_age_days            42.000000
max_financial_age_days              725.000000
Name: value, dtype: float64

### Freeze feature contract

We do not include/select all numeric columns. Keys, raw prices, dates, and target are excluded explicitly. Financial numeric features enter only through Fin_ prefix created by data.py

NaNs from early rolling windows or not-yet-disclosed financials remain here. Their imputation belongs inside a training-only scikit-learn transformer

In [9]:
numeric_features, categorical_features = infer_feature_columns(features_table)
all_features = numeric_features + categorical_features


print("numeric features: ", len(numeric_features))
print("categorical_features: ", len(categorical_features))

display(pd.DataFrame({
    'numeric_feature': pd.Series(numeric_features)
}))
display(pd.DataFrame({
    'categorical_feature': pd.Series(categorical_features)
}))
display(feature_missingness(features_table, all_features).head(30))


assert "Target" not in all_features
assert 'SecuritiesCode' not in all_features
assert 'Date' not in all_features

numeric features:  46
categorical_features:  2


,numeric_feature
0,OpenToCloseReturn
1,HighLowRange
2,GapReturn
3,LogReturn1Day
4,Momentum5Days
5,Momentum20Days
6,Momentum60Days
7,Volatility20Days
8,Volatility60Days
9,VolatilityRatio20To60


,categorical_feature
0,Month
1,DayOfWeek


,missing_count,missing_rate,dtype
Fin_ResultDividendPerShareAnnual_SignedLog,190384,0.081621,float64
Fin_ForecastOperatingProfit_SignedLog,174812,0.074945,float64
Fin_BookValuePerShare_SignedLog,166316,0.071303,float64
Fin_ForecastNetSales_SignedLog,116545,0.049965,float64
Fin_OperatingMargin,115993,0.049728,float64
Fin_OperatingProfit_SignedLog,115444,0.049493,float64
Fin_ForecastDividendPerShareAnnual_SignedLog,110514,0.047379,float64
Fin_ForecastOrdinaryProfit_SignedLog,101473,0.043503,float64
Fin_ForecastEarningsPerShare_SignedLog,94830,0.040655,float64
VolatilityRatio20To60,90192,0.038667,float64


In [10]:
features_table[features_table.isna()]

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,...,Fin_EquityToAssetRatio_SignedLog,Fin_BookValuePerShare_SignedLog,Fin_ResultDividendPerShareAnnual_SignedLog,Fin_ForecastDividendPerShareAnnual_SignedLog,Fin_ForecastNetSales_SignedLog,Fin_ForecastOperatingProfit_SignedLog,Fin_ForecastOrdinaryProfit_SignedLog,Fin_ForecastProfit_SignedLog,Fin_ForecastEarningsPerShare_SignedLog,FinancialAgeDays
0,NaN,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2332526,NaN,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2332527,NaN,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2332528,NaN,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2332529,NaN,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
RANK_FEATURES = [
    "Return1DayRankPct",
    "Momentum20DaysRankPct",
    "Volatility20DaysRankPct",
]

FORBIDDEN_FEATURES = {
    "Target",
    "Date",
    "SecuritiesCode",
    "Open",
    "High",
    "Low",
    "Close",
    "CloseForFeatures",
    "CausalPriceIndex",
    "IssuedSharesLog",
    "MarketCapitalizationLog",
    "Universe0",
    "33SectorCode",
    "17SectorCode",
    "NewMarketSegment",
    "NewIndexSeriesSizeCode",
}

assert len(numeric_features) == len(set(numeric_features))
assert len(categorical_features) == len(set(categorical_features))
assert not set(numeric_features) & set(categorical_features)
assert not set(all_features) & FORBIDDEN_FEATURES

assert not features_table.duplicated(
    ["Date", "SecuritiesCode"]
).any()

assert features_table[RANK_FEATURES].apply(
    lambda column: column.dropna().between(0, 1).all()
).all()

future_financial_report = (
    features_table["FinancialAvailableDate"].notna()
    & features_table["FinancialAvailableDate"].gt(
        features_table["Date"]
    )
)

print(
    "Rows without a previous financial report:",
    features_table["FinancialAvailableDate"].isna().sum(),
)

print(
    "Rows containing a future financial report:",
    future_financial_report.sum(),
)

assert not future_financial_report.any()

assert "output" not in features_table.columns
assert "LogTurnOver" not in features_table.columns

print("Final feature-table validation passed.")

Rows without a previous financial report: 46455
Rows containing a future financial report: 0
Final feature-table validation passed.


In [13]:
labeled = features_table.loc[features_table['Target'].notna()]

correlations = labeled[numeric_features + ['Target']].corr(numeric_only=True)['Target'].drop('Target').sort_values(key=lambda values: values.abs(), ascending=False)

display(correlations.head(15).to_frame("correlation_with_target"))

,correlation_with_target
HasExpectedDividend,-0.040290
ExpectedDividend,-0.035479
MarketWideNoTradeFlag,0.026911
HighLowRange,0.019662
VolumeVs20Days,-0.015457
NoTradeFlag,0.014339
Fin_TotalAssets_SignedLog,-0.012810
Fin_NetSales_SignedLog,-0.012751
LogTurnover,-0.011965
Fin_Equity_SignedLog,-0.011710


## Save feature table

In [16]:
from operator import index


features_path = paths.processed / "features_table.parquet"
features_table.to_parquet(features_path, index=False)

audit = {
    'table_keys': ['Date', 'SecuritiesCode']
    ,'target': 'Target'
    ,'numeric_features': numeric_features
    ,'categorical_features': categorical_features
    ,'financial_lag_days': config.financial_lag_days
    ,'rows': len(features_table)
    ,'date_min': features_table['Date'].min()
    ,'date_max': features_table['Date'].max()
}

save_json(audit, paths.reports / "features_audit.json")
print("saved ", features_path.relative_to(PROJECT_ROOT))

saved  data\processed\features_table.parquet
